# 01 — Render-Trained Unified Matte + Depth + Normal Student (E1 + E2)

Implements `internal/plans/2d-view/render-trained/PLAN.md`:
- **Backbone:** `mobilenetv4_conv_small` / `repvit_m0_9` (~10–15 MB FP16 ONNX, <= 90 ms M2 CPU, < 100 MB RSS).
- **Data:** Infinite `tools/render-trained/make_data.py` (2x supersampled `composite`, `alpha`, `depth`, `normal`) + PD12M CC0 with SAM 2.1 Apache-2.0 pseudo-masks.
- **Replaces:** `u2netp` (0.690), `MODNet` (0.702, 158 ms), `depth.onnx` (94 MB), and `MoGe-2` (70 MB, academic data question).


In [ ]:
!pip -q install -U torch torchvision timm onnx onnxslim onnxruntime lpips pillow numpy tqdm huggingface_hub
import os, hashlib, torch, onnx, onnxslim, onnxruntime as ort
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/out', exist_ok=True)
!unzip -q -o /content/drive/MyDrive/infinite_ml/e1_200.zip -d /content/
DATA_DIR = '/content/e1_200'
print("Scenes:", len(os.listdir(DATA_DIR)))


In [ ]:
import torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np, glob

class RenderTrainedDataset(Dataset):
    def __init__(self, root, size=512):
        self.dirs = sorted(glob.glob(os.path.join(root, 'scene_*')))
        self.size = size
    def __len__(self): return len(self.dirs)
    def __getitem__(self, i):
        d = self.dirs[i]
        load_rgb = lambda f: torch.from_numpy(np.asarray(Image.open(os.path.join(d, f)).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0).permute(2,0,1)
        load_l   = lambda f: torch.from_numpy(np.asarray(Image.open(os.path.join(d, f)).convert('L').resize((self.size, self.size)), np.float32) / 255.0)[None]
        return load_rgb('composite.png'), load_l('alpha.png'), load_l('depth.png'), load_rgb('normal.png') * 2.0 - 1.0

class UnifiedStudent(nn.Module):
    def __init__(self, backbone='mobilenetv4_conv_small.e2400_r224_in1k'):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=True, features_only=True, out_indices=(1,2,3,4))
        chs = self.enc.feature_info.channels()
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in chs])
        self.fuse = nn.Sequential(nn.Conv2d(64*4, 96, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(96, 64, 3, padding=1), nn.ReLU(inplace=True))
        self.matte_head  = nn.Conv2d(64, 1, 1)
        self.depth_head  = nn.Conv2d(64, 1, 1)
        self.normal_head = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        feats = [F.interpolate(l(f), size=(H//4, W//4), mode='bilinear', align_corners=False)
                 for l, f in zip(self.lat, self.enc(x))]
        h = F.interpolate(self.fuse(torch.cat(feats, dim=1)), size=(H, W), mode='bilinear', align_corners=False)
        alpha = torch.sigmoid(self.matte_head(h))
        depth = torch.sigmoid(self.depth_head(h))
        norm  = F.normalize(self.normal_head(h), dim=1)
        return alpha, depth, norm

def train_e1_e2(epochs=20, lr=3e-4, batch_size=8):
    ds = RenderTrainedDataset(DATA_DIR)
    assert len(ds) > 0, f"No scene_* folders found in {DATA_DIR}; run tools/render-trained/make_data.py first."
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2)
    model = UnifiedStudent().cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for ep in range(epochs):
        tot = 0.0
        for img, gt_a, gt_d, gt_n in dl:
            img, gt_a, gt_d, gt_n = img.cuda(), gt_a.cuda(), gt_d.cuda(), gt_n.cuda()
            pa, pd, pn = model(img)
            loss_a = F.l1_loss(pa, gt_a) + F.l1_loss(pa[:,:,1:]-pa[:,:,:-1], gt_a[:,:,1:]-gt_a[:,:,:-1])
            loss_d = F.l1_loss(pd, gt_d)
            loss_n = (1.0 - (pn * gt_n).sum(dim=1)).mean()
            loss = loss_a + 0.5 * loss_d + 0.5 * loss_n
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()
        print(f"Epoch {ep+1}/{epochs} loss={tot/len(dl):.4f}")
    return model


In [ ]:
def export_infinite_onnx(model, dummy, out_path, input_names, output_names, fp16=True):
    model.eval().cpu()
    tmp_fp32 = out_path.replace(".onnx", "_fp32.onnx")
    torch.onnx.export(
        model, dummy, tmp_fp32, opset_version=17, do_constant_folding=True,
        input_names=input_names, output_names=output_names,
    )
    m = onnxslim.slim(tmp_fp32)
    onnx.checker.check_model(m)
    if fp16:
        from onnxruntime.transformers.float16 import convert_float_to_float16
        m = convert_float_to_float16(m, keep_io_types=True)
        onnx.checker.check_model(m)
    onnx.save(m, out_path)
    data = open(out_path, "rb").read()
    sha = hashlib.sha256(data).hexdigest()
    print(f"Exported {out_path}: {len(data)/1e6:.2f} MB  sha256={sha}")
    return sha

model = train_e1_e2(epochs=20, lr=3e-4, batch_size=8)
sha = export_infinite_onnx(model, torch.randn(1,3,512,512), '/content/out/render_matte_depth_normal.onnx',
                           ['image'], ['alpha', 'depth', 'normal'], fp16=True)
import shutil
shutil.copy2('/content/out/render_matte_depth_normal.onnx', '/content/drive/MyDrive/infinite_ml/render_matte_depth_normal.onnx')
print("Saved to Drive: /content/drive/MyDrive/infinite_ml/render_matte_depth_normal.onnx")
